In [49]:
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import pandas as pd



In [50]:
xgdf_train = pd.read_parquet('/Users/Georgi/Dropbox/File group 4B/train_df_20241211.parquet', engine='pyarrow')
xgdf_test = pd.read_parquet('/Users/Georgi/Dropbox/File group 4B/test_df_20241211.parquet', engine='pyarrow')
xgdf_val = pd.read_parquet('/Users/Georgi/Dropbox/File group 4B/val_df_20241211.parquet', engine='pyarrow')

xgdf = pd.concat([xgdf_train, xgdf_test, xgdf_val], axis=0, ignore_index=True)

In [51]:
xgdf_train.head()

,store_nbr,item_nbr,date,onpromotion,store_type,store_cluster,item_family,item_class,perishable,year,week_number_cum,unit_sales
0,1,105857,2014-08-18,0,D,13,GROCERY I,1092,0,2014,86,24.000000
1,1,105857,2014-08-25,0,D,13,GROCERY I,1092,0,2014,87,56.000000
2,1,105857,2014-09-01,0,D,13,GROCERY I,1092,0,2014,88,44.000000
3,1,105857,2014-09-08,0,D,13,GROCERY I,1092,0,2014,89,52.000000
4,1,105857,2014-09-15,0,D,13,GROCERY I,1092,0,2014,90,48.785713


In [52]:
# Define the specific item and store numbers you're interested in
# item_number = 105857
# store_number = 1

# Filter the dataset
# filtered_df = xgdf[(xgdf['store_nbr'] == 1) & (xgdf['store_nbr'] == 2)  ]
filtered_df = xgdf
# Show the filtered dataset


# store_numbers = [1, 2,3,4,5,6]

# Filter the dataset
# filtered_df = xgdf[xgdf['store_nbr'].isin(store_numbers)]

# Show the filtered dataset
print(filtered_df)

         store_nbr  item_nbr       date  onpromotion store_type  \
0                1    105857 2014-08-18            0          D   
1                1    105857 2014-08-25            0          D   
2                1    105857 2014-09-01            0          D   
3                1    105857 2014-09-08            0          D   
4                1    105857 2014-09-15            0          D   
...            ...       ...        ...          ...        ...   
2375251         54   1324670 2017-07-10            2          C   
2375252         54   1324670 2017-07-17            3          C   
2375253         54   1324670 2017-07-24            0          C   
2375254         54   1324670 2017-07-31            0          C   
2375255         54   1324670 2017-08-07            0          C   

         store_cluster item_family  item_class  perishable  year  \
0                   13   GROCERY I        1092           0  2014   
1                   13   GROCERY I        1092           0 

This is a running model in batches! 

Additive Model without "RMSE" calculations in Batches

In [92]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import os
from tqdm import tqdm
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Path for saving batch predictions
output_dir = '/Users/Georgi/Dropbox/File group 4B/'
predictions_file_template = os.path.join(output_dir, 'predictions_batch_{}.parquet')

# Assuming you have the dataset in `filtered_df`
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
df = filtered_df.set_index('date')

# Predefined parameters
alpha = 0.2
beta = 0.1
gamma = 0.2
seasonal_periods = 52  # Assuming weekly seasonality
split_week = 215  # Cumulative week number for train-validation split

# Group by store and item
store_item_groups = list(df.groupby(['store_nbr', 'item_nbr']))
batch_size = len(store_item_groups) // 10  # Divide into 4 equal batches
batches = [store_item_groups[i:i + batch_size] for i in range(0, len(store_item_groups), batch_size)]

# Process each batch
for batch_index, batch in enumerate(batches):
    print(f"Processing batch {batch_index + 1} of {len(batches)}")
    
    # Initialize batch-specific predictions storage
    batch_predictions = []

    # Progress bar for the batch
    with tqdm(total=len(batch), desc=f"Batch {batch_index + 1}", unit="group") as pbar:
        for (store_nbr, item_nbr), group in batch:
            # Split the data based on cumulative week number (<= 189 for train, > 189 for test)
            train = group[group['week_number_cum'] <= split_week]  # Train set: weeks <= 189
            test = group[(group['week_number_cum'] > split_week) ]  # Test set: weeks > 189

            # Initialize storage for t+2 predictions
            t_plus_2_predictions = []
            test_dates = test.index

            # Rolling training data
            rolling_train = train.copy()

            # Iteratively forecast t+2
            for i in range(len(test)):
                # Fit the model on the rolling training data
                model = ExponentialSmoothing(
                    rolling_train['unit_sales'],
                    seasonal_periods=seasonal_periods,
                    trend='add',
                    seasonal='add'
                )
                fitted_model = model.fit(
                    smoothing_level=alpha,
                    smoothing_slope=beta,
                    smoothing_seasonal=gamma,
                    optimized=False
                )

                # Forecast t+2 (second step ahead)
                forecast = fitted_model.forecast(2)[-1]  # Get the second step forecast
                t_plus_2_predictions.append(forecast)

                # Add the observed test value to the rolling training data
                rolling_train = pd.concat([rolling_train, test.iloc[[i]]])

            # Combine the test dates with the predictions and actual sales into a DataFrame
            predictions_df = pd.DataFrame({
                'store_nbr': store_nbr,
                'item_nbr': item_nbr,
                'date': test_dates,
                'actual_sales': test['unit_sales'].values,
                't+2_prediction': t_plus_2_predictions
            })

            # Append the predictions for this item-store combination to the batch list
            batch_predictions.append(predictions_df)

            # Update the progress bar
            pbar.update(1)

    # Export batch predictions
    batch_predictions_df = pd.concat(batch_predictions, ignore_index=True)
    batch_predictions_df.to_parquet(predictions_file_template.format(batch_index + 1), index=False, engine='fastparquet')

# Merge all batch files after processing
all_predictions = pd.concat(
    [pd.read_parquet(predictions_file_template.format(i + 1)) for i in range(len(batches))],
    ignore_index=True
)

# Save the final combined results
all_predictions.to_parquet(os.path.join(output_dir, 'all_predictions.parquet'), index=False, engine='fastparquet')

print("Batch processing complete. Combined results saved.")


Processing batch 1 of 11


Batch 1: 100%|██████████| 1522/1522 [02:47<00:00,  9.10group/s]


Processing batch 2 of 11


Batch 2: 100%|██████████| 1522/1522 [02:49<00:00,  8.99group/s]


Processing batch 3 of 11


...

Processing batch 10 of 11


Batch 10: 100%|██████████| 1522/1522 [02:50<00:00,  8.93group/s]


Processing batch 11 of 11


Batch 11: 100%|██████████| 6/6 [00:00<00:00,  8.76group/s]

Batch processing complete. Combined results saved.


Multiplicative model in batches 

In [ ]:
# import numpy as np
# import pandas as pd
# from statsmodels.tsa.holtwinters import ExponentialSmoothing
# import os
# from tqdm import tqdm
# import warnings

# # Suppress warnings
# warnings.filterwarnings('ignore')

# # Path for saving batch predictions and RMSE results
# output_dir = '/Users/Georgi/Dropbox/File group 4B/'
# predictions_file_template = os.path.join(output_dir, 'predictions_batch_{}.parquet')
# rmse_file_template = os.path.join(output_dir, 'rmse_batch_{}.parquet')

# # Assuming you have the dataset in `filtered_df`
# filtered_df['date'] = pd.to_datetime(filtered_df['date'])
# df = filtered_df.set_index('date')

# # Handle zeros in the 'unit_sales' column by replacing them with a small constant (e.g., 0.1)
# filtered_df['unit_sales'] = filtered_df['unit_sales'].replace(0, 0.1)

# # Predefined parameters
# alpha = 0.1
# beta = 0.1
# gamma = 0.1
# seasonal_periods = 52  # Assuming weekly seasonality
# split_week = 189  # Cumulative week number for train-test split

# # Group by store and item
# store_item_groups = list(df.groupby(['store_nbr', 'item_nbr']))
# batch_size = len(store_item_groups) // 10  # Divide into 4 equal batches
# batches = [store_item_groups[i:i + batch_size] for i in range(0, len(store_item_groups), batch_size)]

# # Process each batch
# for batch_index, batch in enumerate(batches):
#     print(f"Processing batch {batch_index + 1} of {len(batches)}")
    
#     # Initialize batch-specific RMSE and predictions storage
#     batch_rmse_results = []
#     batch_predictions = []

#     # Track the current store number being processed
#     completed_stores = set()

#     # Progress bar for the batch
#     with tqdm(total=len(batch), desc=f"Batch {batch_index + 1}", unit="group") as pbar:
#         for (store_nbr, item_nbr), group in batch:
#             # Split the data based on cumulative week number (<= 189 for train, > 189 for test)
#             train = group[group['week_number_cum'] <= split_week]  # Train set: weeks <= 189
#             test = group[(group['week_number_cum'] > split_week) & (group['week_number_cum'] <= 215)]  # Test set: weeks > 189

#             # Initialize storage for t+2 predictions
#             t_plus_2_predictions = []
#             test_dates = test.index

#             # Rolling training data
#             rolling_train = train.copy()

#             # Iteratively forecast t+2
#             for i in range(len(test)):
#                 # Fit the model on the rolling training data
#                 model = ExponentialSmoothing(
#                     rolling_train['unit_sales'],
#                     seasonal_periods=seasonal_periods,
#                     trend='mul',
#                     seasonal='mul'
#                 )
#                 fitted_model = model.fit(
#                     smoothing_level=alpha,
#                     smoothing_slope=beta,
#                     smoothing_seasonal=gamma,
#                     optimized=False
#                 )

#                 # Forecast t+2 (second step ahead)
#                 forecast = fitted_model.forecast(2)[-1]  # Get the second step forecast
#                 t_plus_2_predictions.append(forecast)

#                 # Add the observed test value to the rolling training data
#                 rolling_train = pd.concat([rolling_train, test.iloc[[i]]])

#             # Combine the test dates with the predictions and actual sales into a DataFrame
#             predictions_df = pd.DataFrame({
#                 'store_nbr': store_nbr,
#                 'item_nbr': item_nbr,
#                 'date': test_dates,
#                 'actual_sales': test['unit_sales'].values,
#                 't+2_prediction': t_plus_2_predictions
#             })

#             # Append the predictions for this item-store combination to the batch list
#             batch_predictions.append(predictions_df)

#             # Add the store number to the completed list if all its items are processed
#             completed_stores.add(store_nbr)

#             # Update the progress bar
#             pbar.update(1)

#         # Notify which stores were completed in this batch
#         print(f"Completed stores in batch {batch_index + 1}: {sorted(completed_stores)}")

#     # Export batch predictions and RMSE results
#     batch_predictions_df = pd.concat(batch_predictions, ignore_index=True)
#     batch_predictions_df.to_parquet(predictions_file_template.format(batch_index + 1), index=False, engine='fastparquet')

#     # Merge all batch files after processing
# all_predictions = pd.concat(
#     [pd.read_parquet(predictions_file_template.format(i + 1)) for i in range(len(batches))],
#     ignore_index=True
# )

# # Save the final combined results
# all_predictions.to_parquet(os.path.join(output_dir, 'all_predictionsMUL.parquet'), index=False, engine='fastparquet')

# print("Batch processing complete. Combined results saved.")


In [93]:
predictions_df = pd.read_parquet('/Users/Georgi/Dropbox/File group 4B/all_predictions.parquet')

unique_stores = predictions_df['store_nbr'].unique()
print(unique_stores)

predictions_df

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 15 16 19 23 24 27 30 32 33 34 35
 37 38 39 40 41 44 45 46 47 48 49 50 51 53 54]


,store_nbr,item_nbr,date,actual_sales,t+2_prediction
0,1,105857,2017-02-13,20.000000,15.552215
1,1,105857,2017-02-20,16.290476,14.039663
2,1,105857,2017-02-27,16.000000,25.937650
3,1,105857,2017-03-06,25.428572,43.780607
4,1,105857,2017-03-13,24.666668,22.332503
...,...,...,...,...,...
395871,54,1324670,2017-07-10,24.000000,27.639636
395872,54,1324670,2017-07-17,33.000000,33.367145
395873,54,1324670,2017-07-24,22.142857,31.815273
395874,54,1324670,2017-07-31,22.000000,33.750247


In [94]:
# Calculate RMSE for each store-item combination
rmse_per_group = predictions_df.groupby(['store_nbr', 'item_nbr']).apply(
    lambda group: np.sqrt(((group['actual_sales'] - group['t+2_prediction']) ** 2).mean())
)

# Calculate the average RMSE across all store-item combinations
average_rmse = rmse_per_group.mean()

# Print the average RMSE
print(f"Average RMSE across all store-item combinations: {average_rmse}")


Average RMSE across all store-item combinations: 32.35891200404011
